# ARC-AGI-3 DuckWADL — Integrated AI Difference Learning + DifferenceFusion

**Core loop:** `observe → build current-game difference state → compare EXPLOIT vs EXPLORE → fuse evidence → act once → measure actual delta → update ADL → next move`.

This build integrates ADL directly into the Duck action loop. It preserves the strict single-environment/no-cross-game-prior contract while adding explicit difference signatures, weighted DifferenceFusion, prediction calibration, current-game difference memory, and post-run coverage/quality auditing.


In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Fixed real-run configuration

Run every discovered game with concurrency 4 and strict no-prior behavior. A dynamic per-game cap keeps both complete passes inside Kaggle's nine-hour GPU runtime limit. No environment can be omitted.


In [ ]:
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY

# Optional grafts are constrained to the same current game and current run.
# Banking and transfer are intentionally never installed.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(
        os.environ.get("TAAF_CONTEXT_WINDOW", "32768")
    ),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags
print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    "games_required=all_discovered "
    f"source_per_game_budget={_original_game_budget}"
)


## 7. Integrated DuckWADL ADL + DifferenceFusion policy

This cell upgrades the Duck analyzer from prompt-only dual-path selection into an explicit ADL architecture: a current-game Difference Memory schema, semantic difference signatures, weighted DifferenceFusion, prediction calibration, and mandatory post-move learning after every committed action. Candidate A/B comparison remains internal; only the selected action touches the real environment.


In [ ]:
# === DUCKWADL HARD-WIRED LIVE ADL v4 — PYTHON-ENFORCED AFTER EVERY REAL MOVE ===
import copy
import hashlib
import json
import math
import os
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional
from inference.agent.tool_agent import ToolAgent

DUCKWADL_ADL_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2
ADL_SCHEMA = "adl.arc3.duckwadl.hardwired-live.v4"

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DIFFERENCE_MEMORY_LOG = WORKING_DIR / "adl_difference_memory.jsonl"
LIVE_CONTROLLER_LOG = WORKING_DIR / "adl_live_controller.jsonl"

ADL_FUSION_WEIGHTS = {
    "legality": 0.15,
    "predicted_progress": 0.18,
    "predicted_frame_change": 0.10,
    "information_gain": 0.13,
    "novelty": 0.10,
    "causal_consistency": 0.12,
    "world_model_consistency": 0.10,
    "action_efficiency": 0.07,
    "loop_avoidance": 0.05,
}
assert abs(sum(ADL_FUSION_WEIGHTS.values()) - 1.0) < 1e-9

# Only a material evidence gap can veto the model-selected discrete action.
# Mouse/coordinate actions are never rewritten by this controller.
ADL_VETO_MARGIN = float(os.environ.get("DUCKWADL_ADL_VETO_MARGIN", "0.18"))
ADL_MIN_CONTEXT_SAMPLES_FOR_VETO = int(os.environ.get("DUCKWADL_ADL_MIN_VETO_SAMPLES", "2"))
ADL_MAX_MEMORY_CONTEXT = int(os.environ.get("DUCKWADL_ADL_CONTEXT_RECORDS", "10"))


def _clip01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def _clip11(x: float) -> float:
    return max(-1.0, min(1.0, float(x)))


def _stable_json(value: Any) -> str:
    def clean(v):
        if isinstance(v, dict):
            return {str(k): clean(val) for k, val in sorted(v.items(), key=lambda kv: str(kv[0]))
                    if str(k).lower() not in {"timestamp", "time", "elapsed", "latency", "request_id"}}
        if isinstance(v, (list, tuple)):
            return [clean(x) for x in v]
        if isinstance(v, (str, int, float, bool)) or v is None:
            return v
        return str(v)
    return json.dumps(clean(value), sort_keys=True, separators=(",", ":"), ensure_ascii=True)


def _digest(value: Any, n: int = 16) -> str:
    return hashlib.sha256(_stable_json(value).encode("utf-8", errors="replace")).hexdigest()[:n]


def _read_state_payload(state_path: Path) -> Dict[str, Any]:
    try:
        raw = Path(state_path).read_text(encoding="utf-8", errors="replace")
        value = json.loads(raw)
        return value if isinstance(value, dict) else {"value": value}
    except Exception as exc:
        return {"unreadable_state": type(exc).__name__, "path": str(state_path)}


def _recursive_number(payload: Any, keys: set[str]) -> Optional[float]:
    if isinstance(payload, dict):
        for k, v in payload.items():
            if str(k).lower() in keys and isinstance(v, (int, float)) and not isinstance(v, bool):
                return float(v)
        for v in payload.values():
            found = _recursive_number(v, keys)
            if found is not None:
                return found
    elif isinstance(payload, (list, tuple)):
        for v in payload:
            found = _recursive_number(v, keys)
            if found is not None:
                return found
    return None


def _state_signature(state_payload: Dict[str, Any]) -> str:
    return _digest(state_payload, 20)


def _normalize_action_name(value: Any) -> str:
    text = str(value or "").strip()
    if not text:
        return ""
    return text.upper()


def _extract_action_name(action_payload: Any) -> str:
    if isinstance(action_payload, str):
        return _normalize_action_name(action_payload)
    if not isinstance(action_payload, dict):
        return _normalize_action_name(action_payload)
    for key in ("action", "action_name", "name", "type"):
        if key in action_payload:
            candidate = _normalize_action_name(action_payload.get(key))
            if candidate.startswith("ACTION") or candidate in {"RESET", "MOUSE"}:
                return candidate
    # Some ARC callbacks carry a one-key action dict.
    if len(action_payload) == 1:
        only = next(iter(action_payload))
        candidate = _normalize_action_name(only)
        if candidate.startswith("ACTION") or candidate in {"RESET", "MOUSE"}:
            return candidate
    return ""


def _replace_action_name(action_payload: Any, new_name: str) -> Any:
    new_name = _normalize_action_name(new_name)
    if isinstance(action_payload, str):
        return new_name
    if not isinstance(action_payload, dict):
        return action_payload
    payload = copy.deepcopy(action_payload)
    for key in ("action", "action_name", "name", "type"):
        if key in payload:
            old = _normalize_action_name(payload.get(key))
            if old.startswith("ACTION") or old in {"RESET", "MOUSE"}:
                payload[key] = new_name
                return payload
    # Do not invent a schema when the callback shape is unknown.
    return action_payload


@dataclass
class ADLDifferenceSignature:
    step: int
    action: str
    state_signature: str = ""
    next_state_signature: str = ""
    state_changed: str = "uncertain"
    score_delta: Optional[float] = None
    level_delta: Optional[float] = None
    prediction_match: str = "uncertain"
    information_gain: float = 0.0
    progress_value: float = 0.0
    loop_signal: bool = False
    novel_transition: str = "uncertain"
    lesson: str = ""
    next_bias: str = "neutral"
    evidence_strength: float = 0.0
    model_selected_action: str = ""
    controller_executed_action: str = ""
    controller_vetoed: bool = False
    controller_score: float = 0.0

    def validate(self) -> None:
        if self.step < 0:
            raise ValueError("ADL step must be non-negative")
        self.information_gain = _clip01(self.information_gain)
        self.progress_value = _clip11(self.progress_value)
        self.evidence_strength = _clip01(self.evidence_strength)
        self.controller_score = _clip01(self.controller_score)
        if self.next_bias not in {"exploit", "explore", "neutral"}:
            raise ValueError("next_bias must be exploit/explore/neutral")


class CurrentGameDifferenceMemory:
    """Strictly current-game ADL ledger used live by the Python controller."""
    def __init__(self, game_id: str):
        self.game_id = str(game_id)
        self.records: List[ADLDifferenceSignature] = []
        self.by_context_action: Dict[str, List[ADLDifferenceSignature]] = {}
        self.by_action: Dict[str, List[ADLDifferenceSignature]] = {}
        self.visited_state_counts: Dict[str, int] = {}
        self.transition_counts: Dict[str, int] = {}
        self.prediction_hits = 0.0
        self.prediction_total = 0

    @staticmethod
    def _key(state_signature: str, action: str) -> str:
        return f"{state_signature}|{_normalize_action_name(action)}"

    def add(self, record: ADLDifferenceSignature) -> None:
        record.validate()
        self.records.append(record)
        key = self._key(record.state_signature, record.action)
        self.by_context_action.setdefault(key, []).append(record)
        self.by_action.setdefault(record.action, []).append(record)
        if record.next_state_signature:
            self.visited_state_counts[record.next_state_signature] = self.visited_state_counts.get(record.next_state_signature, 0) + 1
        transition_key = f"{record.state_signature}|{record.action}|{record.next_state_signature}"
        self.transition_counts[transition_key] = self.transition_counts.get(transition_key, 0) + 1
        if record.prediction_match in {"yes", "partial", "no"}:
            self.prediction_total += 1
            self.prediction_hits += {"yes": 1.0, "partial": 0.5, "no": 0.0}[record.prediction_match]

    def context_records(self, state_signature: str, action: str) -> List[ADLDifferenceSignature]:
        return self.by_context_action.get(self._key(state_signature, action), [])

    def action_records(self, action: str) -> List[ADLDifferenceSignature]:
        return self.by_action.get(_normalize_action_name(action), [])

    def empirical_action_value(self, action: str, state_signature: str = "") -> float:
        local = self.context_records(state_signature, action) if state_signature else []
        rows = local or self.action_records(action)
        if not rows:
            return 0.0
        weights = [max(0.05, r.evidence_strength) for r in rows]
        return sum(r.progress_value * w for r, w in zip(rows, weights)) / sum(weights)

    def repeated_noop_rate(self, state_signature: str, action: str) -> float:
        rows = self.context_records(state_signature, action)
        if not rows:
            return 0.0
        noops = sum(r.state_changed == "no" for r in rows)
        return noops / len(rows)

    def loop_rate(self, state_signature: str, action: str) -> float:
        rows = self.context_records(state_signature, action)
        if not rows:
            return 0.0
        return sum(bool(r.loop_signal) for r in rows) / len(rows)

    def prediction_accuracy(self) -> float:
        return self.prediction_hits / self.prediction_total if self.prediction_total else 0.0

    def novelty_for_action(self, state_signature: str, action: str) -> float:
        n = len(self.context_records(state_signature, action))
        return 1.0 / (1.0 + n)

    def controller_metrics(self, state_signature: str, action: str, legal: bool = True) -> Dict[str, float]:
        local = self.context_records(state_signature, action)
        global_rows = self.action_records(action)
        empirical = self.empirical_action_value(action, state_signature)
        progress01 = _clip01((empirical + 1.0) / 2.0)
        noop = self.repeated_noop_rate(state_signature, action)
        loop = self.loop_rate(state_signature, action)
        novelty = self.novelty_for_action(state_signature, action)
        sample_strength = _clip01(len(local) / 3.0)
        changed_rate = 0.5
        if local:
            known = [r for r in local if r.state_changed in {"yes", "no"}]
            if known:
                changed_rate = sum(r.state_changed == "yes" for r in known) / len(known)
        info = 0.5
        if local:
            info = sum(r.information_gain for r in local) / len(local)
        elif global_rows:
            info = sum(r.information_gain for r in global_rows[-8:]) / min(8, len(global_rows))
        causal = 0.5 + 0.5 * sample_strength if empirical > 0 else 0.5 * (1.0 - sample_strength * max(0.0, -empirical))
        return {
            "legality": 1.0 if legal else 0.0,
            "predicted_progress": progress01,
            "predicted_frame_change": _clip01(changed_rate),
            "information_gain": _clip01(max(info, novelty * 0.65)),
            "novelty": _clip01(novelty),
            "causal_consistency": _clip01(causal),
            "world_model_consistency": _clip01(0.5 + 0.5 * sample_strength if empirical >= 0 else 0.5 - 0.4 * sample_strength),
            "action_efficiency": _clip01(1.0 - noop),
            "loop_avoidance": _clip01(1.0 - loop),
        }

    def controller_score(self, state_signature: str, action: str, legal: bool = True) -> float:
        return adl_fusion_utility(self.controller_metrics(state_signature, action, legal=legal))

    def rank_actions(self, state_signature: str, valid_actions: List[str]) -> List[Dict[str, Any]]:
        rows = []
        for action in valid_actions:
            name = _normalize_action_name(action)
            metrics = self.controller_metrics(state_signature, name, legal=True)
            rows.append({
                "action": name,
                "score": adl_fusion_utility(metrics),
                "samples": len(self.context_records(state_signature, name)),
                "empirical_progress": self.empirical_action_value(name, state_signature),
                "noop_rate": self.repeated_noop_rate(state_signature, name),
                "loop_rate": self.loop_rate(state_signature, name),
                "metrics": metrics,
            })
        rows.sort(key=lambda r: (r["score"], r["empirical_progress"], -r["noop_rate"]), reverse=True)
        return rows

    def compact_context(self, state_signature: str, valid_actions: List[str], limit: int = ADL_MAX_MEMORY_CONTEXT) -> str:
        ranking = self.rank_actions(state_signature, valid_actions)
        recent = self.records[-max(1, int(limit)):]
        payload = {
            "schema": ADL_SCHEMA,
            "game_id": self.game_id,
            "state_signature": state_signature,
            "records_seen": len(self.records),
            "prediction_accuracy": round(self.prediction_accuracy(), 4),
            "controller_ranking": ranking,
            "recent_differences": [asdict(r) for r in recent],
        }
        return json.dumps(payload, sort_keys=True, separators=(",", ":"))


def adl_fusion_utility(metrics: Dict[str, float]) -> float:
    total = 0.0
    for key, weight in ADL_FUSION_WEIGHTS.items():
        value = _clip01(metrics.get(key, 0.0))
        total += weight * value
    return _clip01(total)


def _infer_transition_record(*, step: int, pre_state: Dict[str, Any], post_state: Dict[str, Any], result: Any,
                             model_action: str, executed_action: str, controller_vetoed: bool,
                             controller_score: float, memory: CurrentGameDifferenceMemory) -> ADLDifferenceSignature:
    pre_sig = _state_signature(pre_state)
    post_sig = _state_signature(post_state)
    changed = pre_sig != post_sig
    score_before = _recursive_number(pre_state, {"score", "reward", "final_score"})
    score_after = _recursive_number(post_state, {"score", "reward", "final_score"})
    if score_after is None:
        score_after = _recursive_number(result, {"score", "reward", "final_score"})
    level_before = _recursive_number(pre_state, {"level", "levels_completed", "level_index"})
    level_after = _recursive_number(post_state, {"level", "levels_completed", "level_index"})
    if level_after is None:
        level_after = _recursive_number(result, {"level", "levels_completed", "level_index"})
    score_delta = (score_after - score_before) if score_before is not None and score_after is not None else None
    level_delta = (level_after - level_before) if level_before is not None and level_after is not None else None
    transition_key = f"{pre_sig}|{executed_action}|{post_sig}"
    repeated_transition = memory.transition_counts.get(transition_key, 0)
    destination_visits = memory.visited_state_counts.get(post_sig, 0)
    loop_signal = bool((not changed) or repeated_transition >= 1 or (post_sig == pre_sig) or destination_visits >= 3)
    novelty = repeated_transition == 0 and destination_visits == 0

    progress = 0.0
    if score_delta is not None:
        progress += max(-0.7, min(0.7, score_delta))
    if level_delta is not None and level_delta > 0:
        progress += 0.8
    if not changed:
        progress -= 0.25
    if loop_signal:
        progress -= 0.20
    if changed and score_delta in {None, 0.0} and (level_delta in {None, 0.0}):
        progress += 0.08
    progress = _clip11(progress)

    info_gain = 0.0
    if novelty and changed:
        info_gain = 0.8
    elif changed:
        info_gain = 0.35
    if level_delta is not None and level_delta > 0:
        info_gain = max(info_gain, 0.6)
    if not changed:
        info_gain = 0.05

    # Python cannot know the model's semantic prediction exactly without intercepting
    # model content pre-dispatch, so prediction_match remains uncertainty-aware.
    prediction_match = "uncertain"
    evidence = 0.35
    if score_delta is not None or level_delta is not None:
        evidence += 0.20
    if changed != (pre_sig == post_sig):
        evidence += 0.10
    if repeated_transition >= 1:
        evidence += 0.15
    evidence = _clip01(evidence)

    if progress > 0.15:
        next_bias = "exploit"
    elif progress < -0.10 or info_gain > 0.55:
        next_bias = "explore"
    else:
        next_bias = "neutral"

    lesson = (
        f"{pre_sig[:8]} + {executed_action} -> {post_sig[:8]}; "
        f"changed={changed}; progress={progress:.3f}; info={info_gain:.3f}; loop={loop_signal}"
    )
    return ADLDifferenceSignature(
        step=int(step),
        action=executed_action,
        state_signature=pre_sig,
        next_state_signature=post_sig,
        state_changed="yes" if changed else "no",
        score_delta=score_delta,
        level_delta=level_delta,
        prediction_match=prediction_match,
        information_gain=info_gain,
        progress_value=progress,
        loop_signal=loop_signal,
        novel_transition="yes" if novelty else "no",
        lesson=lesson,
        next_bias=next_bias,
        evidence_strength=evidence,
        model_selected_action=model_action,
        controller_executed_action=executed_action,
        controller_vetoed=bool(controller_vetoed),
        controller_score=controller_score,
    )


DUCKWADL_ADL_CONTRACT = r"""
DUCKWADL HARD-WIRED LIVE ADL v4

Python now maintains a current-game DifferenceMemory and supplies a controller ranking before
this analyzer turn. Use that ranking as empirical evidence, not as an oracle. You still form
exactly two candidates from the SAME current observation: A=EXPLOIT and B=EXPLORE.

Before the real tool action, visibly emit DUAL_PATH_DECISION with both candidates, their
predictions, component scores and SELECT. Then issue exactly ONE real environment action.

After the tool returns, visibly emit POST_MOVE_ADL. Separately, the Python callback wrapper
will independently hash the real pre/post runtime state, compute no-op/loop/novelty/progress
signals, and append them to current-game memory before the next analyzer turn.

The Python controller may veto a discrete model-selected action ONLY when current-state
empirical evidence is sufficiently repeated and another legal discrete action exceeds it by
the configured margin. It never rewrites MOUSE/coordinate actions, never executes a second
candidate, never forks the environment, and never uses cross-game memory.

STRICT BOUNDARY: current game/current run only. No historical routes, prior submissions,
replays, hidden labels, source-code introspection, or cross-game ADL transfer.
""".strip()


class DuckWADLToolAgent(ToolAgent):
    """ToolAgent whose actual step_env callback is wrapped by live Python ADL."""
    def __init__(self, game_id: str = "unknown", **kwargs):
        super().__init__(**kwargs)
        self._adl_memory = CurrentGameDifferenceMemory(game_id)
        self._adl_game_id = str(game_id)
        self._adl_turn = 0
        if DUCKWADL_ADL_CONTRACT not in self._system_prompt:
            self._system_prompt = self._system_prompt.rstrip() + "\n\n" + DUCKWADL_ADL_CONTRACT

    def _controller_context(self, state_path: Path, valid_actions: List[str]) -> tuple[str, str]:
        payload = _read_state_payload(state_path)
        sig = _state_signature(payload)
        context = self._adl_memory.compact_context(sig, valid_actions)
        return sig, context

    def _choose_controller_override(self, *, state_signature: str, requested_action: str,
                                    valid_actions: List[str]) -> tuple[str, bool, List[Dict[str, Any]]]:
        requested = _normalize_action_name(requested_action)
        normalized_valid = [_normalize_action_name(a) for a in valid_actions if _normalize_action_name(a)]
        ranking = self._adl_memory.rank_actions(state_signature, normalized_valid)
        if not requested or requested not in normalized_valid or not ranking:
            return requested, False, ranking
        # Never rewrite coordinate-sensitive or reset semantics.
        if requested in {"MOUSE", "RESET"}:
            return requested, False, ranking
        best = ranking[0]
        requested_row = next((r for r in ranking if r["action"] == requested), None)
        if requested_row is None:
            return requested, False, ranking
        best_name = best["action"]
        if best_name in {"MOUSE", "RESET"} or best_name == requested:
            return requested, False, ranking
        # Require repeated evidence about at least one of the competing actions at this state.
        evidence_samples = max(int(best["samples"]), int(requested_row["samples"]))
        margin = float(best["score"] - requested_row["score"])
        harmful_repeat = requested_row["noop_rate"] >= 0.5 or requested_row["loop_rate"] >= 0.5
        if evidence_samples >= ADL_MIN_CONTEXT_SAMPLES_FOR_VETO and margin >= ADL_VETO_MARGIN and harmful_repeat:
            return best_name, True, ranking
        return requested, False, ranking

    def analyze(self, state_path: Path, action_num: int, valid_actions: List[str] | None = None,
                step_env: Callable[[Dict[str, Any]], Dict[str, Any]] | None = None, **kwargs):
        valid_actions = list(valid_actions or [])
        state_sig, controller_context = self._controller_context(state_path, valid_actions)
        live_addendum = (
            "\n\n[PYTHON_ADL_CONTROLLER_CURRENT_GAME]\n"
            + controller_context
            + "\n[/PYTHON_ADL_CONTROLLER_CURRENT_GAME]\n"
            + "Use this current-game empirical ranking when estimating A/B DifferenceFusion."
        )
        original_prompt = self._system_prompt
        self._system_prompt = original_prompt.rstrip() + live_addendum

        wrapped_step = step_env
        if step_env is not None:
            def wrapped_step(action_payload):
                pre_state = _read_state_payload(state_path)
                pre_sig = _state_signature(pre_state)
                requested = _extract_action_name(action_payload)
                executed, vetoed, ranking = self._choose_controller_override(
                    state_signature=pre_sig,
                    requested_action=requested,
                    valid_actions=valid_actions,
                )
                actual_payload = _replace_action_name(action_payload, executed) if vetoed else action_payload
                actual_name = _extract_action_name(actual_payload) or requested or executed or "UNKNOWN"
                controller_score = self._adl_memory.controller_score(pre_sig, actual_name, legal=True) if actual_name else 0.0

                result = step_env(actual_payload)
                post_state = _read_state_payload(state_path)
                record = _infer_transition_record(
                    step=int(action_num),
                    pre_state=pre_state,
                    post_state=post_state,
                    result=result,
                    model_action=requested,
                    executed_action=actual_name,
                    controller_vetoed=vetoed,
                    controller_score=controller_score,
                    memory=self._adl_memory,
                )
                self._adl_memory.add(record)
                event = {
                    "schema": ADL_SCHEMA,
                    "game_id": self._adl_game_id,
                    "action_num": int(action_num),
                    "requested_action": requested,
                    "executed_action": actual_name,
                    "vetoed": vetoed,
                    "ranking": ranking,
                    "record": asdict(record),
                }
                with LIVE_CONTROLLER_LOG.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(event, sort_keys=True) + "\n")
                with DIFFERENCE_MEMORY_LOG.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps({"game_id": self._adl_game_id, **asdict(record)}, sort_keys=True) + "\n")
                print(
                    f"[PYTHON_ADL][UPDATE] game={self._adl_game_id} step={action_num} "
                    f"requested={requested or '?'} executed={actual_name} veto={vetoed} "
                    f"changed={record.state_changed} progress={record.progress_value:.3f} "
                    f"loop={record.loop_signal} memory={len(self._adl_memory.records)}",
                    flush=True,
                )
                return result
            wrapped_step = wrapped_step

        try:
            # Preserve ToolAgent's exact analyzer contract; only the callback and prompt are wrapped.
            return super().analyze(
                state_path=state_path,
                action_num=action_num,
                valid_actions=valid_actions,
                step_env=wrapped_step,
                **kwargs,
            )
        finally:
            self._system_prompt = original_prompt
            self._adl_turn += 1


def _duckwadl_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "vrfai/Qwen3.6-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    game_id = getattr(game, "game_id", None) or getattr(game, "id", None) or getattr(game, "env_name", None) or f"game-{index}"
    return DuckWADLToolAgent(
        game_id=str(game_id),
        model=model,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
        base_url=base_url,
        provider="vllm",
    )


bm.solver.analyzer_factory = _duckwadl_adl_analyzer_factory

print("DUCKWADL HARD-WIRED LIVE ADL v4 ACTIVE", flush=True)
print(f"ADL_SCHEMA={ADL_SCHEMA}", flush=True)
print("LIVE HOOK: ToolAgent.analyze step_env callback", flush=True)
print("BEFORE MOVE: Python current-state ranking injected into analyzer context", flush=True)
print("AFTER MOVE: Python hashes pre/post state and DifferenceMemory.add() runs synchronously", flush=True)
print(f"VETO POLICY: harmful repeated discrete action only; margin={ADL_VETO_MARGIN:.3f}; min_samples={ADL_MIN_CONTEXT_SAMPLES_FOR_VETO}", flush=True)
print("MOUSE/RESET REWRITE: DISABLED", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME DIFFERENCE MEMORY: DISABLED", flush=True)


## 7.1 ADL integration self-test

Validates DifferenceFusion normalization, difference-record constraints, and current-game memory behavior before any environment action is executed.


In [ ]:
# === HARD-WIRED LIVE ADL SELF-TEST: NO ENVIRONMENT ACTIONS ===
_test_memory = CurrentGameDifferenceMemory("self-test")
_pre = {"frame": [[0, 0], [0, 1]], "score": 0, "level": 1}
_post = {"frame": [[0, 0], [1, 0]], "score": 0, "level": 1}
_sig = _state_signature(_pre)
for i in range(2):
    rec = _infer_transition_record(
        step=i,
        pre_state=_pre,
        post_state=_pre,  # repeated no-op deliberately teaches a penalty
        result={"score": 0, "level": 1},
        model_action="ACTION1",
        executed_action="ACTION1",
        controller_vetoed=False,
        controller_score=0.5,
        memory=_test_memory,
    )
    _test_memory.add(rec)
rec2 = _infer_transition_record(
    step=2,
    pre_state=_pre,
    post_state=_post,
    result={"score": 0, "level": 1},
    model_action="ACTION2",
    executed_action="ACTION2",
    controller_vetoed=False,
    controller_score=0.5,
    memory=_test_memory,
)
rec2.progress_value = 0.5
rec2.evidence_strength = 0.8
_test_memory.add(rec2)
_rank = _test_memory.rank_actions(_sig, ["ACTION1", "ACTION2"])
assert len(_test_memory.records) == 3
assert _test_memory.repeated_noop_rate(_sig, "ACTION1") == 1.0
assert _test_memory.loop_rate(_sig, "ACTION1") == 1.0
assert _rank[0]["action"] == "ACTION2", _rank
assert 0.0 <= adl_fusion_utility({k: 0.5 for k in ADL_FUSION_WEIGHTS}) <= 1.0
assert _extract_action_name({"action": "ACTION3"}) == "ACTION3"
assert _extract_action_name("ACTION4") == "ACTION4"
print("HARD-WIRED ADL SELF-TEST PASSED")
print("learned ranking:", json.dumps(_rank, indent=2, sort_keys=True))
del _test_memory, _pre, _post, _sig, _rank, rec, rec2


## 8. Run exactly one real environment trajectory per game

Competition and local modes share the same policy. Competition mode discovers games
from the official gateway. Local mode uses the mounted public `environment_files`.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    )
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
RUN_PER_GAME_SECONDS = min(
    1500.0,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "schema": "adl.arc3.dual-path.clean.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "DUAL-PATH RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f}",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "DUAL-PATH SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)


## 10. Final ADL run summary


In [ ]:
# === FINAL CLEAN ADL SUMMARY ===
import json
import re

runs = list(bm.game_runs)
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]

summary = {
    "schema": "adl.arc3.dual-path.clean.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(runs),
    "mean_score": (sum(scores) / len(scores) if scores else 0.0),
    "positive_score_games": sum(score > 0 for score in scores),
    "total_levels_completed": sum(levels),
    "total_actions": sum(actions),
    "concurrency": TARGET_CONCURRENCY,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "second_environment_pass": False,
    "strict_no_prior": True,
    "submission_path": str(SUBMISSION_PATH),
}

summary_path = WORKING_DIR / "dual_path_adl_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("=" * 72)
print("ARC-AGI-3 DUAL-PATH ADL — CLEAN RUN")
print(f"competition_rerun={TRUE_SUBMISSION}")
print("2 internal plans -> 1 selected action -> 1 environment trajectory")
print("SECOND ENVIRONMENT PASS: DISABLED")
print(
    f"games={summary['games']} "
    f"mean_score={summary['mean_score']:.6f} "
    f"positive_games={summary['positive_score_games']} "
    f"levels={summary['total_levels_completed']} "
    f"actions={summary['total_actions']}"
)
print(f"submission={SUBMISSION_PATH}")
print(f"summary={summary_path}")
print("=" * 72)


## 11. Structured DuckWADL ADL audit

Parses the visible per-move DifferenceFusion and `POST_MOVE_ADL` traces into a current-run difference dataset, measures coverage against committed actions, and reports prediction calibration, progress, information gain, loops, and novel transitions. The generated difference-memory file is an audit artifact and is **not** loaded across games.


In [ ]:
# === DUCKWADL STRUCTURED ADL EXTRACTION / AUDIT ===
# Converts visible PLAN/POST_MOVE_ADL traces from THIS run into a structured
# difference-learning dataset, while verifying that ADL covered the real actions.

import json
import re
from pathlib import Path

_TEXT_EXTS = {".log", ".txt", ".json", ".jsonl", ".md"}
_SKIP_NAMES = {
    "dual_path_adl_summary.json",
    "post_move_adl_audit.json",
    "duckwadl_adl_audit.json",
    "adl_difference_memory.jsonl",
}


def _adl_text_files(root: Path):
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in _TEXT_EXTS:
            continue
        if path.name in _SKIP_NAMES:
            continue
        yield path


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _extract_blocks(text: str, marker: str):
    # Capture marker payload until next blank boundary/major tagged event.
    pattern = re.compile(
        re.escape(marker) + r"\s*\n(?P<body>.*?)(?=\n\[(?:DUCKWADL|DIFFERENCEFUSION)\]|\n(?:DUAL_PATH_DECISION:|POST_MOVE_ADL:)|\Z)",
        re.S,
    )
    return [m.group("body") for m in pattern.finditer(text)]


def _parse_key_values(body: str):
    out = {}
    for line in body.splitlines():
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip().upper()
        value = value.strip()
        if key:
            out[key] = value
    return out


plan_records = []
post_records = []
files_scanned = []
for path in _adl_text_files(WORKING_DIR):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    plan_bodies = _extract_blocks(text, "DUAL_PATH_DECISION:")
    post_bodies = _extract_blocks(text, "POST_MOVE_ADL:")
    if plan_bodies or post_bodies:
        files_scanned.append(str(path))
    plan_records.extend(_parse_key_values(body) for body in plan_bodies)
    post_records.extend(_parse_key_values(body) for body in post_bodies)

# De-duplicate repeated logger copies conservatively by key fields.
def _dedupe(records, keys):
    seen = set()
    unique = []
    for rec in records:
        sig = tuple(rec.get(k, "") for k in keys)
        if sig in seen:
            continue
        seen.add(sig)
        unique.append(rec)
    return unique

plan_records = _dedupe(plan_records, ("STEP", "A_ACTION", "B_ACTION", "SELECT"))
post_records = _dedupe(post_records, ("STEP", "ACTION", "DIFFERENCE_SIGNATURE", "LESSON"))

# Emit an explicit current-run ADL dataset. It is an audit artifact only; it is
# never loaded into another game by this notebook.
PARSED_ADL_LOG = WORKING_DIR / "adl_llm_post_move_parsed.jsonl"
with PARSED_ADL_LOG.open("w", encoding="utf-8") as f:
    for rec in post_records:
        row = {
            "schema": ADL_SCHEMA,
            "step": int(rec["STEP"]) if rec.get("STEP", "").isdigit() else rec.get("STEP"),
            "action": rec.get("ACTION"),
            "state_changed": rec.get("STATE_CHANGED"),
            "state_delta": rec.get("STATE_DELTA"),
            "score_delta": _safe_float(rec.get("SCORE_DELTA")),
            "level_delta": _safe_float(rec.get("LEVEL_DELTA")),
            "prediction_match": rec.get("PREDICTION_MATCH"),
            "information_gain": _safe_float(rec.get("INFORMATION_GAIN")),
            "progress_value": _safe_float(rec.get("PROGRESS_VALUE")),
            "loop_signal": rec.get("LOOP_SIGNAL"),
            "novel_transition": rec.get("NOVEL_TRANSITION"),
            "difference_signature": rec.get("DIFFERENCE_SIGNATURE"),
            "lesson": rec.get("LESSON"),
            "evidence_strength": _safe_float(rec.get("EVIDENCE_STRENGTH")),
            "next_bias": rec.get("NEXT_BIAS"),
        }
        f.write(json.dumps(row, sort_keys=True) + "\n")

runs = list(getattr(bm, "game_runs", []) or [])
total_actions = sum(len(getattr(run, "history", ()) or ()) for run in runs)

matches = [r.get("PREDICTION_MATCH", "").lower() for r in post_records]
known_predictions = [m for m in matches if m in {"yes", "partial", "no"}]
exact_prediction_accuracy = (
    sum(m == "yes" for m in known_predictions) / len(known_predictions)
    if known_predictions else None
)
soft_prediction_accuracy = (
    sum(1.0 if m == "yes" else 0.5 if m == "partial" else 0.0 for m in known_predictions)
    / len(known_predictions)
    if known_predictions else None
)
progress_values = [
    value for value in (_safe_float(r.get("PROGRESS_VALUE")) for r in post_records)
    if value is not None
]
info_values = [
    value for value in (_safe_float(r.get("INFORMATION_GAIN")) for r in post_records)
    if value is not None
]

post_move_coverage = len(post_records) / total_actions if total_actions else 0.0
plan_coverage = len(plan_records) / total_actions if total_actions else 0.0

live_controller_rows = []
if LIVE_CONTROLLER_LOG.exists():
    for line in LIVE_CONTROLLER_LOG.read_text(encoding="utf-8", errors="ignore").splitlines():
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            live_controller_rows.append(row)
live_memory_rows = []
if DIFFERENCE_MEMORY_LOG.exists():
    for line in DIFFERENCE_MEMORY_LOG.read_text(encoding="utf-8", errors="ignore").splitlines():
        try:
            row = json.loads(line)
        except Exception:
            continue
        if isinstance(row, dict):
            live_memory_rows.append(row)
live_veto_count = sum(bool(r.get("vetoed")) for r in live_controller_rows)
live_update_coverage = len(live_memory_rows) / total_actions if total_actions else 0.0

audit = {
    "schema": ADL_SCHEMA,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(runs),
    "total_actions": total_actions,
    "live_python_adl_updates": len(live_memory_rows),
    "live_python_adl_coverage": live_update_coverage,
    "live_controller_vetoes": live_veto_count,
    "dual_path_decisions": len(plan_records),
    "post_move_adl_updates": len(post_records),
    "plan_coverage": plan_coverage,
    "post_move_coverage": post_move_coverage,
    "prediction_exact_accuracy": exact_prediction_accuracy,
    "prediction_soft_accuracy": soft_prediction_accuracy,
    "mean_progress_value": sum(progress_values) / len(progress_values) if progress_values else None,
    "mean_information_gain": sum(info_values) / len(info_values) if info_values else None,
    "loop_signals": sum(str(r.get("LOOP_SIGNAL", "")).lower() == "yes" for r in post_records),
    "novel_transitions": sum(str(r.get("NOVEL_TRANSITION", "")).lower() == "yes" for r in post_records),
    "llm_post_move_rows": len(post_records),
    "llm_parsed_path": str(PARSED_ADL_LOG),
    "live_difference_memory_path": str(DIFFERENCE_MEMORY_LOG),
    "live_controller_path": str(LIVE_CONTROLLER_LOG),
    "files_with_adl_trace": sorted(set(files_scanned)),
    "single_environment_pass": True,
    "cross_game_memory": False,
    "required_policy": "pre-action DifferenceFusion + POST_MOVE_ADL after every committed real move",
}

audit_path = WORKING_DIR / "duckwadl_adl_audit.json"
audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("=" * 72)
print("DUCKWADL ADL AUDIT")
print(json.dumps(audit, indent=2, sort_keys=True))
print(f"live_difference_memory={DIFFERENCE_MEMORY_LOG}")
print(f"llm_parsed_memory={PARSED_ADL_LOG}")
print(f"live_controller={LIVE_CONTROLLER_LOG}")
print(f"audit={audit_path}")
print("=" * 72)
